In [13]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from network.net_autoencoder import Encoder, Decoder
from network.net_fusion import FusionNetworkConcat

In [14]:
# ==========================================================
# 1️⃣ 图像加载函数（记录原始尺寸）
# ==========================================================
def load_ir_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape[:2]
    img_resized = cv2.resize(img, (256, 256))
    tensor = torch.tensor(img_resized / 255.0, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    return tensor, img, (h, w)

def load_visible_image(path):
    img_bgr = cv2.imread(path)
    h, w = img_bgr.shape[:2]
    img_bgr_resized = cv2.resize(img_bgr, (256, 256))
    img_lab = cv2.cvtColor(img_bgr_resized, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    l_tensor = torch.tensor(l / 255.0, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    return l_tensor, a, b, img_bgr, (h, w)


# ==========================================================
# 2️⃣ 融合结果重建彩色图像
# ==========================================================
def reconstruct_color_image(fused_L_tensor, a_channel, b_channel, orig_size):
    fused_L = fused_L_tensor.squeeze().detach().cpu().numpy()

    # === 自动归一化亮度到 [0,1] ===
    fused_L = (fused_L - fused_L.min()) / (fused_L.max() - fused_L.min() + 1e-8)

    # === 恢复到原始尺寸 ===
    fused_L = cv2.resize(fused_L, (orig_size[1], orig_size[0]))
    fused_L_uint8 = (fused_L * 255).astype(np.uint8)

    # === ⚠️ 将 a、b 通道同步 resize 回原始尺寸 ===
    a_resized = cv2.resize(a_channel, (orig_size[1], orig_size[0]))
    b_resized = cv2.resize(b_channel, (orig_size[1], orig_size[0]))

    # === 合并成 LAB 并转回 RGB ===
    fused_lab = cv2.merge([fused_L_uint8, a_resized, b_resized])
    fused_bgr = cv2.cvtColor(fused_lab, cv2.COLOR_LAB2BGR)
    fused_rgb = cv2.cvtColor(fused_bgr, cv2.COLOR_BGR2RGB)

    return Image.fromarray(fused_rgb)

In [15]:

# ==========================================================
# 3️⃣ 推理函数（自动尺寸恢复 + 自动亮度校正）
# ==========================================================
def test_fusion_with_pretrained(ir_encoder_path, vi_encoder_path, decoder_path,
                                ir_image_path, vi_image_path, save_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"▶ 使用设备: {device}")

    # --- 1. 创建网络结构 ---
    encoder_ir = Encoder(in_channels=1)
    encoder_vi = Encoder(in_channels=1)
    fusion_shallow = FusionNetworkConcat(in_channels=32)   # 浅层融合
    fusion_deep = FusionNetworkConcat(in_channels=128)     # 深层融合
    decoder = Decoder(in_channels=128)

    # --- 2. 加载权重 ---
    encoder_ir.load_state_dict(torch.load(ir_encoder_path, map_location=device))
    encoder_vi.load_state_dict(torch.load(vi_encoder_path, map_location=device))
    decoder.load_state_dict(torch.load(decoder_path, map_location=device))

    encoder_ir.to(device).eval()
    encoder_vi.to(device).eval()
    fusion_shallow.to(device).eval()
    fusion_deep.to(device).eval()
    decoder.to(device).eval()

    # --- 3. 读取图像 ---
    ir_tensor, ir_img, orig_size_ir = load_ir_image(ir_image_path)
    vi_L_tensor, vi_a, vi_b, vi_rgb, orig_size_vi = load_visible_image(vi_image_path)

    # 确保两张图的原始尺寸一致（如果不同，用可见光为主）
    orig_size = orig_size_vi

    ir_tensor, vi_L_tensor = ir_tensor.to(device), vi_L_tensor.to(device)

    # --- 4. 前向推理 ---
    with torch.no_grad():
        ir_conv, ir_dense = encoder_ir(ir_tensor)
        vi_conv, vi_dense = encoder_vi(vi_L_tensor)

        fused_conv = fusion_shallow(ir_conv, vi_conv)
        fused_dense = fusion_deep(ir_dense, vi_dense)

        fused_L = decoder(fused_conv, fused_dense)
        fused_L = torch.clamp(fused_L, 0, 1)

    print(f"✅ 融合完成, 输出尺寸: {list(fused_L.shape)}")

    # --- 5. 重建彩色图像（自动归一化+恢复尺寸） ---
    fused_rgb = reconstruct_color_image(fused_L, vi_a, vi_b, orig_size)

    # --- 6. 保存输出 ---
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fused_rgb.save(save_path)
    print(f"💾 融合彩色图像已保存到: {save_path}")

    return fused_rgb



In [16]:
# ==============================================================
# 4️⃣ 主程序入口
# ==============================================================
if __name__ == "__main__":
    # 模型权重路径（根据你的文件路径调整）
    ir_encoder_path = r"../weights/encoder_final.pth"
    vi_encoder_path = r"../weights/encoder_final.pth"
    decoder_path = r"../weights/decoder_final.pth"

    # 测试图像路径
    ir_image_path = r"../image/testNet/250256_ir.jpg"
    vi_image_path = r"../image/testNet/250256_vi.jpg"

    # 输出路径
    save_path = r"../output/fusion/fused_rgb_concat.png"

    fused_img = test_fusion_with_pretrained(
        ir_encoder_path, vi_encoder_path, decoder_path,
        ir_image_path, vi_image_path, save_path
    )

    fused_img.show()

▶ 使用设备: cuda
[Encoder] conv1_out: torch.Size([1, 32, 256, 256])
[Encoder] dense_out: torch.Size([1, 128, 32, 32])
[Encoder] conv1_out: torch.Size([1, 32, 256, 256])
[Encoder] dense_out: torch.Size([1, 128, 32, 32])
✅ 融合完成, 输出尺寸: [1, 1, 256, 256]


C:\Users\DELL\AppData\Local\Temp\ipykernel_22776\4035301515.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoder_ir.load_state_dict(torch.load(ir_encoder_path, map_l

💾 融合彩色图像已保存到: ../output/fusion/fused_rgb_concat.png
